# Aula 01: Fundamentos de PyTorch e Tensores

**Disciplina:** Tópicos A - Redes Neurais Profundas (2026)
**Prof:** Thiago Medeiros

---

Bem-vindos! Este notebook é o ponto de partida para nossa jornada. Não vamos apenas "rodar código", vamos entender a estrutura de dados fundamental do Deep Learning: o **Tensor**.

## Objetivos do Lab:
1.  Dominar a criação e manipulação de tensores (Shapes, Dtypes, Devices).
2.  Entender o conceito de **Broadcasting** (essencial para evitar loops for).
3.  Manipular dimensões com `view`, `reshape`, `permute`.
4.  Implementar uma operação matricial eficiente na GPU.

---

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 1. Anatomia de um Tensor

Um tensor no PyTorch é muito semelhante a um array `numpy`, mas com suporte a aceleração via GPU. Todo tensor possui três atributos críticos:
- **torch.dtype**: O tipo de dado (float32, int64, bool...).
- **torch.device**: Onde ele vive (cpu, cuda:0).
- **torch.layout**: Como ele está na memória (strided, sparse).

In [ ]:
# Criação básica
t_float = torch.tensor([[1.0, 2.0], [3.0, 4.0]], dtype=torch.float32)
t_int   = torch.tensor([1, 2, 3], dtype=torch.int64)

print(f"Tensor Float:\n{t_float}")
print(f"Shape: {t_float.shape} | Type: {t_float.dtype} | Device: {t_float.device}")

### 1.1 Construtores Comuns
Evite criar listas python gigantes para depois converter. Use os construtores nativos:

In [ ]:
x_zeros = torch.zeros(3, 3)
x_ones  = torch.ones(2, 4)
x_rand  = torch.randn(5, 5) # Distribuição Normal (média 0, var 1)
x_seq   = torch.arange(0, 10, step=0.5)

print("Sequência:", x_seq)

## 2. Operações e Broadcasting

O broadcasting é a mágica que permite operar tensores de tamanhos diferentes. Ele segue regras estritas.

**Exemplo:** Queremos normalizar uma matriz (subtrair a média de cada coluna).

In [ ]:
# Dados: 10 amostras, 3 features
data = torch.randn(10, 3)

# Média de cada coluna (dim=0 colapsa as linhas)
mean = data.mean(dim=0)
print(f"Data shape: {data.shape}")
print(f"Mean shape: {mean.shape}")

# Broadcasting acontece aqui:
# (10, 3) - (3) -> O PyTorch 'estica' o (3) para transformar em (1, 3) e depois repete 10 vezes para virar (10, 3)
centered_data = data - mean

print("\nPrimeiras 3 linhas centralizadas:")
print(centered_data[:3])

# Verificação: a nova média deve ser próxima de 0
print("\nNova média das colunas (deve ser ~0):", centered_data.mean(dim=0))

## 3. Manipulação de Formas (Reshape/View)

Em Redes Neurais, passamos o tempo todo redimensionando dados (ex: converter uma imagem 28x28 em um vetor chato de 784 para entrar numa MLP).

- `.view()`: Retorna uma visualização nova dos mesmos dados (rápido, mas requer contiguidade).
- `.reshape()`: Similar ao view, mas copia os dados se necessário (mais seguro).
- `.permute()`: Troca a ordem das dimensões.

In [ ]:
# Imagine um batch de 64 imagens coloridas 32x32
batch_images = torch.randn(64, 3, 32, 32) # (N, C, H, W)

# 1. Flattening para MLP: (N, 3*32*32)
mlp_input = batch_images.view(64, -1)
print(f"MLP Input Shape: {mlp_input.shape}")

# 2. Permutation: Mudando para formato (N, H, W, C) - comum em matplotlib
plt_format = batch_images.permute(0, 2, 3, 1)
print(f"Matplotlib Format Shape: {plt_format.shape}")

## 4. Prática: Regressão Linear Manual

Vamos implementar uma regressão linear sem usar o módulo `torch.nn`, apenas com operações de tensor. 

Modelo: $y = w \cdot x + b$

In [ ]:
# 1. Dados Sintéticos
X = torch.linspace(0, 10, 100).view(-1, 1) # (100, 1)
noise = torch.randn_like(X) * 2
y_true = 3 * X + 10 + noise # w=3, b=10

plt.scatter(X, y_true, s=10)
plt.title("Dados Sintéticos")
plt.show()

In [ ]:
# 2. Inicialização de Parâmetros (com Gradients Habilitados!)
w = torch.randn(1, 1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

lr = 0.01

# 3. Loop de Treinamento
for epoch in range(100):
    # Forward
    y_pred = X @ w + b # @ é multiplicação matricial
    
    # Loss (MSE)
    loss = ((y_pred - y_true)**2).mean()
    
    # Backward (Calcula gradientes)
    loss.backward()
    
    # Update (Desabilita tracking de gradiente para não bugar o gráfico)
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
        
        # Zera gradientes para próxima iteração
        w.grad.zero_()
        b.grad.zero_()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}")

print(f"\nResultado Final: w={w.item():.2f}, b={b.item():.2f}")
print(f"Valores Reais:   w=3.00, b=10.00")

## Desafio para Casa
Adapte o código acima para rodar na GPU (se disponível). Lembre-se que X, y_true, w e b precisam estar no mesmo dispositivo.